# GPEN-BFR-256 — Face Restoration

Generative blind face restoration at 256². It synthesises plausible detail (does not recover exact pixels).

**Model:** `gpen_bfr_256.onnx`  |  **Feature:** Blind face restoration (256²)

Runs on CPU or GPU. Cells: setup → data → load → test → benchmark → accuracy → training notes → copyright.


## 1. Setup - code + dependencies


In [ ]:
import os, sys, subprocess, time, io, urllib.request, platform
from pathlib import Path
REPO = 'https://github.com/Zubair-hussain/Xovato-cpu-ai-studio.git'
if Path('app').exists(): ROOT = Path('.').resolve()
elif Path('../app').exists(): ROOT = Path('..').resolve()
else:
    subprocess.run(['git','clone','--depth','1',REPO], check=True)
    ROOT = Path('Xovato-cpu-ai-studio').resolve()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','pillow','numpy','onnxruntime','soundfile'], check=False)
print('root:', ROOT, '| python', platform.python_version())

## 2. Data (test set)


In [ ]:
print('Needs a real face + yunet + gpen. Set FACE_IMAGE to a portrait path.')
FACE_IMAGE = ''
from pathlib import Path as _P
have = bool(FACE_IMAGE) and _P(FACE_IMAGE).exists()

## 3. Download & load the model


In [ ]:
def fetch(url, dest, min_bytes):
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size >= min_bytes:
        print('present', dest.name, f'{dest.stat().st_size/1e6:.1f}MB'); return dest
    print('downloading', dest.name, '...'); urllib.request.urlretrieve(url, dest)
    assert dest.stat().st_size >= min_bytes, 'file too small (LFS pointer?)'
    print('ok', f'{dest.stat().st_size/1e6:.1f}MB'); return dest

yunet = fetch('https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx', ROOT/'models'/'face'/'yunet.onnx', 100_000)
gpen = fetch('https://github.com/facefusion/facefusion-assets/releases/download/models-3.0.0/gpen_bfr_256.onnx', ROOT/'models'/'face'/'gpen_bfr_256.onnx', 20_000_000)

## 4. Inference test


In [ ]:
from app import image_processing as ip
if have:
    data=_P(FACE_IMAGE).read_bytes()
    payload, analysis = ip.restore_faces_image(data, detector_path=yunet, restorer_path=gpen)
    print(analysis.summary); from PIL import Image; Image.open(io.BytesIO(payload))
else: print('Set FACE_IMAGE to a real (ideally low-quality) portrait.')

## 5. Benchmark (latency)


In [ ]:
if have:
    data=_P(FACE_IMAGE).read_bytes(); N=3; t=time.perf_counter()
    for _ in range(N): ip.restore_faces_image(data, detector_path=yunet, restorer_path=gpen)
    print(f'{(time.perf_counter()-t)/N:.2f} s/photo')
else: print('need a face image')

## 6. Accuracy signal


In [ ]:
print('Reported: blind face restoration on par with GFPGAN; strong LPIPS/FID on CelebA-Test (256).')
print('To measure: build LQ/HQ pairs (CelebA-Test), restore, compute PSNR/SSIM/LPIPS/FID.')

## 7. Training / fine-tuning notes

GPEN (Yang et al., CVPR 2021) is a GAN prior trained on **FFHQ** with a degradation model for blind restoration. Retraining needs the GPEN repo and a GPU. This notebook uses the released 256² ONNX for inference.


## 8. Copyright


In [ ]:
# ====================================================================
# COPYRIGHT NOTICE - DO NOT REMOVE
# ====================================================================
# (c) Zubair Hussain - Xovato AI Studio.
# These notebooks, the benchmark/accuracy harness, and the surrounding
# tooling are the work of Zubair Hussain. You may use and modify them for
# your own projects; keep this attribution intact, including in the
# background-removal notebook and any derived outputs.
print('(c) Zubair Hussain - Xovato AI Studio')